# Showcase 05 — travel booking bot (flights + hotels) on graxella

A small, realistic multi-agent system. **Two agents, four tools, one mesh** — and every graxella superpower comes for free. No JSON schemas, no handoff protocols, no A2A envelope wiring. That's the whole thesis: **silent plumbing**.

**What you WRITE**
* 4 `@tool` functions
* 2 agents (one native LangGraph, one `graxella.Agent` — both work)
* 1 constitution invariant *(optional)*
* 1 gate policy *(optional)*
* ONE call: `graxella.mesh([...])`

**What graxella HANDLES silently**
* routing (TF-IDF picks the right agent for each user turn)
* peer-directory injection (each agent knows what the other can do)
* memory (every booking + decision goes into mnema, cross-turn recall)
* constitution (invariants fire on every routed task, detection-only)
* gate (learning proposals scored + auto-approved/rejected)
* `why()` (cross-source provenance for any decision)

**Prereqs:** Ollama daemon running with `qwen2.5:3b` pulled and listening on `localhost:11434`.

## Setup — put the `graxella` package on the path

In [1]:
from __future__ import annotations

import sys
import tempfile
import uuid
from pathlib import Path

# When executed from the showcase dir, hop one level up to import graxella.
_REPO = Path.cwd().resolve()
for candidate in [_REPO, *_REPO.parents]:
    if (candidate / 'graxella' / '__init__.py').exists():
        sys.path.insert(0, str(candidate))
        break

from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

import graxella
from graxella import (Agent, Constitution, GatePolicy, Memory, ObjectiveScores,
                      PromotionGate, UnifiedTracer)

workdir = Path(tempfile.mkdtemp(prefix='graxella-travel-'))
print(f'[setup] workdir: {workdir}')
print('[setup] LLM:     Ollama qwen2.5:3b (must be listening on localhost:11434)')

[setup] workdir: C:\Users\Sridhar\AppData\Local\Temp\graxella-travel-syl8t7fx
[setup] LLM:     Ollama qwen2.5:3b (must be listening on localhost:11434)


## 1 — Tools

Four `@tool` functions backing our mock catalog. In a real system these hit Amadeus / Skyscanner / Booking.com.

In [2]:
_FLIGHT_CATALOG = {
    ('NYC', 'PAR', '2026-12-15'): [
        {'id': 'AF012', 'carrier': 'Air France', 'price_usd': 720, 'depart': '18:30', 'duration': '7h 25m'},
        {'id': 'DL264', 'carrier': 'Delta',       'price_usd': 685, 'depart': '21:15', 'duration': '7h 40m'},
        {'id': 'UA057', 'carrier': 'United',      'price_usd': 899, 'depart': '17:00', 'duration': '7h 10m'},
    ],
}
_HOTEL_CATALOG = {
    ('paris', 'eiffel tower'): [
        {'id': 'HP-9021', 'name': 'Hotel Le Marais',     'nightly_usd': 189, 'walk_min': 22, 'rating': 4.4},
        {'id': 'HP-1188', 'name': 'Champ de Mars Suites','nightly_usd': 275, 'walk_min': 4,  'rating': 4.6},
        {'id': 'HP-4402', 'name': 'Trocadero Boutique',  'nightly_usd': 219, 'walk_min': 11, 'rating': 4.5},
    ],
}

@tool
def search_flights(origin: str, destination: str, date: str) -> list:
    """Search flights for a given origin, destination, and date (YYYY-MM-DD).

    Args:
        origin: 3-letter city code (e.g. NYC).
        destination: 3-letter city code (e.g. PAR).
        date: departure date in YYYY-MM-DD format.
    """
    key = (origin.upper(), destination.upper(), date)
    return _FLIGHT_CATALOG.get(key, [])

@tool
def book_flight(flight_id: str, passenger_name: str) -> dict:
    """Book a specific flight by its id for a named passenger."""
    return {
        'confirmation': f'CONF-{uuid.uuid4().hex[:8].upper()}',
        'flight_id': flight_id,
        'passenger': passenger_name,
        'status': 'confirmed',
    }

@tool
def search_hotels(city: str, near_landmark: str) -> list:
    """Search hotels in a city near a specific landmark."""
    key = (city.lower().strip(), near_landmark.lower().strip())
    return _HOTEL_CATALOG.get(key, [])

@tool
def book_hotel(hotel_id: str, nights: int, guest_name: str) -> dict:
    """Book a specific hotel by its id for N nights under a guest name."""
    return {
        'confirmation': f'HTL-{uuid.uuid4().hex[:8].upper()}',
        'hotel_id': hotel_id,
        'nights': nights,
        'guest': guest_name,
        'status': 'confirmed',
        'requires_review': True,   # trips the constitution invariant
    }
print('tools ready:', [t.name for t in [search_flights, book_flight, search_hotels, book_hotel]])

tools ready: ['search_flights', 'book_flight', 'search_hotels', 'book_hotel']


## 2 — Agents

Two agents, **different shapes, both first-class citizens** in a graxella mesh.

In [3]:
llm = ChatOllama(model='qwen2.5:3b', temperature=0)

# Path A: native LangGraph.
flight_agent = create_react_agent(
    llm,
    tools=[search_flights, book_flight],
    name='flights',
)

# Path B: graxella.Agent (CrewAI-shape). Same wire underneath.
hotel_agent = Agent(
    role='hotels',
    goal='search and book hotels near landmarks the guest asks about',
    backstory=('You help travellers find and book hotels. Always call '
               'search_hotels first to see options, then book_hotel when '
               'the guest picks one. Keep responses short.'),
    tools=[search_hotels, book_hotel],
    llm=llm,
)
print("flights = create_react_agent(llm, [search_flights, book_flight], name='flights')")
print("hotels  = graxella.Agent(role='hotels', tools=[search_hotels, book_hotel], llm=llm)")

flights = create_react_agent(llm, [search_flights, book_flight], name='flights')
hotels  = graxella.Agent(role='hotels', tools=[search_hotels, book_hotel], llm=llm)


C:\Users\Sridhar\AppData\Local\Temp\ipykernel_37868\161086801.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  flight_agent = create_react_agent(


## 3 — Wrap them

One call to `graxella.mesh([...])` wires memory + routing + constitution + gate + tracer. Every agent's system prompt is auto-augmented with a **peer directory**.

In [4]:
memory = Memory.sqlite(db_path=str(workdir / 'mnema.db'),
                       agent_id='travel-bot')
tracer = UnifiedTracer.default()
gate = PromotionGate(
    threshold=0.85, require_human=True,
    policy=GatePolicy(
        weights={'quality': 0.4, 'compliance': 0.3, 'cost': 0.2, 'latency': 0.1},
        cost_reference=0.10, latency_reference=500.0,
        compliance_floor=0.9, auto_approve=0.85, needs_human_min=0.5,
    ),
)
constitution = Constitution.from_dict({
    'version': '1.0',
    'invariants': [{
        'name': 'hotel_bookings.require_confirmation',
        'applies_to': 'delegate', 'severity': 'warning',
        'predicate': {
            'type': 'object',
            'properties': {
                'chosen_agent': {'not': {'const': 'hotels'}},
            },
        },
    }],
})

bot = graxella.mesh(
    [flight_agent, hotel_agent],
    memory=memory,
    tracer=tracer,
    gate=gate,
    constitution=constitution,
    router='tfidf',
    store_path=str(workdir / 'routes.jsonl'),
)
print('agents registered:', bot.society.agents())

agents registered: ['flights', 'hotels']


## 4 — Five-turn conversation

The user talks to the bot. TF-IDF picks the right agent for each turn (zero-LLM routing decisions). Booking IDs persist across turns via mnema.

In [5]:
turns = [
    'Find flights from NYC to PAR on 2026-12-15',
    'Book flight DL264 for John Smith',
    'Find hotels in Paris near the Eiffel Tower',
    'Book hotel HP-9021 for 5 nights for John Smith',
    'Actually, book HP-1188 for 3 nights instead for John Smith',
]
decision_ids: list[str] = []
for i, msg in enumerate(turns, 1):
    print(f'\nturn {i}> {msg}')
    out = bot.invoke({'messages': [('user', msg)]})
    route = out['route']
    print(f"  routed -> {route['agent']}  score={route['score']:.3f}  strategy={route['strategy']}")
    content = out['messages'][-1]['content']
    snip = content if len(content) < 220 else content[:220] + '...'
    print(f'  bot:    {snip}')
    if route.get('decision_id'):
        decision_ids.append(route['decision_id'])


turn 1> Find flights from NYC to PAR on 2026-12-15
  routed -> flights  score=0.667  strategy=tfidf
  bot:    {'result': 'I found some flights for you from NYC to PAR on December 15, 2026. Here are the options:\n\n1. **Flight ID AF012** by Air France:\n   - Departure: 18:30 (6:30 PM)\n   - Duration: 7 hours and 25 minutes\n   - ...

turn 2> Book flight DL264 for John Smith
  routed -> flights  score=0.431  strategy=tfidf
  bot:    {'result': 'The flight DL264 has been booked for John Smith with confirmation number CONF-A3C6C6C7.', 'tool_calls': [{'name': 'book_flight', 'args': {'flight_id': 'DL264', 'passenger_name': 'John Smith'}}]}

turn 3> Find hotels in Paris near the Eiffel Tower
  routed -> hotels  score=0.444  strategy=tfidf
  bot:    {'result': "I found some hotels near the Eiffel Tower in Paris. Here are a few options:\n\n1. **Hotel Le Marais** - Walk distance is approximately 22 minutes from your location. It has a rating of 4.4.\n   - Nightly pric...

turn 4> Book hotel HP-

## 5 — Constitution

The invariant `hotel_bookings.require_confirmation` trips whenever `hotels` is picked. **Detection-only** — bookings still went through.

In [6]:
viols = tracer.events(event_type='governance.constitution_violation')
print(f'{len(viols)} violation(s) recorded (detection-only, bookings still went through)')
for v in viols[:5]:
    p = v.payload
    print(f"  [{p['severity']}] {p['name']}")

3 violation(s) recorded (detection-only, bookings still went through)
  [warning] hotel_bookings.require_confirmation
  [warning] hotel_bookings.require_confirmation
  [warning] hotel_bookings.require_confirmation


## 6 — Gate scoring

Three learning proposals scored against `GatePolicy`. Narrow + high-quality + full-compliance auto-approves; wide blast → needs human; compliance floor breach → auto-reject.

In [7]:
proposals = [
    ('route.tag_add',
     {'agent': 'hotels', 'add_terms': ['vacation', 'trip', 'stay']},
     'narrow',
     ObjectiveScores(cost_usd=0.01, latency_ms=90, quality=0.95, compliance=1.0)),
    ('rule.new',
     {'pattern': 'auto-approve flight bookings under $500'},
     'wide',
     ObjectiveScores(cost_usd=0.01, latency_ms=90, quality=0.95, compliance=1.0)),
    ('skill.new',
     {'agent': 'hotels', 'skill': 'book_hotel_without_review'},
     'narrow',
     ObjectiveScores(cost_usd=0.01, latency_ms=90, quality=0.95, compliance=0.4)),
]
for kind, payload, blast, obj in proposals:
    p = gate.propose(kind, payload, blast_radius=blast, objectives=obj)
    decision, after = gate.auto_evaluate(p.id)
    print(f'#{after.id} {after.kind:<14} blast={after.blast_radius:<7} '
          f'scalar={after.score:.3f} compliance={obj.compliance:.2f} '
          f'-> {decision.value.upper():<13} (status={after.status.value})')

#1 route.tag_add  blast=narrow  scalar=0.942 compliance=1.00 -> NEEDS_HUMAN   (status=pending)
#2 rule.new       blast=wide    scalar=0.942 compliance=1.00 -> NEEDS_HUMAN   (status=pending)
#3 skill.new      blast=narrow  scalar=0.762 compliance=0.40 -> AUTO_REJECT   (status=rejected)


## 7 — `why()` for turn 4 (hotel booking)

Cross-source provenance: tracer chain + mnema assertion + provenance + derived rules and skills.

In [8]:
if len(decision_ids) >= 4:
    joint = bot.why(decision_ids[3])
    print(f"tracer chain events: {len(joint.get('tracer_chain', []))}")
    mnema = joint.get('mnema') or {}
    if isinstance(mnema, dict) and 'error' not in mnema:
        print(f'mnema keys:          {list(mnema.keys())}')
        assertion = mnema.get('assertion') or {}
        if isinstance(assertion, dict):
            print(f"subject:             {assertion.get('subject')}")
            print(f"statement:           {assertion.get('statement')}")

tracer chain events: 2
mnema keys:          ['assertion_id', 'assertion', 'provenance', 'status', 'superseded_by', 'derived_rules', 'derived_skills', 'wal_events']
subject:             decision::delegate::hotels::book_a_specific_hotel_by_its_id_for_n_nights_under_a_guest_name
statement:           [delegate] task='Book hotel HP-9021 for 5 nights for John Smith' chose='hotels::book_a_specific_hotel_by_its_id_for_n_nights_under_a_guest_name' :: hotels::book_a_specific_hotel_by_its_id_for_n_nights_under_a_guest_name selected (score=0.447). Semantic match 0.535 on tokens [book, for, hotel, nights]; tag overlap 0.182 on [book, for, hotel, nights]. Runner-up flights::book_a_specific_flight_by_its_id_for_a_named_passenger scored 0.219 (gap 0.228).


## 8 — Cross-turn memory recall

Every routed turn was recorded, replayable by decision id. The `chosen` column proves the agent + skill picked for each turn is still in mnema.

In [9]:
rep = memory.report()
print(f'mnema report: {rep}')
print(f'{len(decision_ids)} routing decisions -> each replayable via bot.why(id):')
for i, did in enumerate(decision_ids, 1):
    joint = bot.why(did)
    chain = joint.get('tracer_chain', [])
    mnema = joint.get('mnema') or {}
    assertion = mnema.get('assertion') if isinstance(mnema, dict) else None
    chosen = '?'
    conf = 0.0
    if isinstance(assertion, dict):
        subject = assertion.get('subject') or ''
        parts = subject.split('::')
        if len(parts) >= 4 and parts[0] == 'decision':
            chosen = f'{parts[2]}::{parts[3]}'
        conf = (assertion.get('confidence') or {}).get('value') or 0.0
    print(f'  turn {i}: {did[:20]}...  chain={len(chain)}  chosen={chosen:<40} conf={conf:.3f}')

mnema report: {'agent_id': 'travel-bot', 'namespace': 'default', 'generated_at': '2026-08-15T22:12:48.753539+00:00', 'total_assertions_active': 5, 'total_wal_events': 5, 'digest_versions': 0, 'active_rules': [], 'active_skills': [], 'supersession_chains': []}
5 routing decisions -> each replayable via bot.why(id):
  turn 1: asr_0cdfa4bb2104404d...  chain=1  chosen=flights::flights_agent                   conf=0.667
  turn 2: asr_e7bd9ae33b2a41f8...  chain=1  chosen=flights::book_a_specific_flight_by_its_id_for_a_named_passenger conf=0.431
  turn 3: asr_179740c55fd74355...  chain=2  chosen=hotels::search_hotels_in_a_city_near_a_specific_landmark conf=0.444
  turn 4: asr_4df2fef785974e27...  chain=2  chosen=hotels::book_a_specific_hotel_by_its_id_for_n_nights_under_a_guest_name conf=0.447
  turn 5: asr_537d7117d1cd41ce...  chain=2  chosen=hotels::book_a_specific_hotel_by_its_id_for_n_nights_under_a_guest_name conf=0.358


---

## The takeaway

**What you wrote:** 4 tools + 2 agents + 1 `mesh()` call.

**What graxella did silently:** routing, memory, peer-awareness, governance, gate scoring, provenance. Zero JSON schemas.